---

# Fishora: Backbone Linear Probe Comparison

*Same-recipe linear probes over six pretrained backbones, exported for the Fishora evaluation suite.*

---

## Fishora Team

- Member 1
- Member 2
- Member 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Pipeline**](#3)
4. [**Training Recipe**](#4)
5. [**Linear Probe Runs**](#5)
6. [**Results**](#6)
7. [**Export**](#7)


---

# Introduction <a name="1"></a>

---

## Overview

*Fishora identifies landed fish from a photograph taken at a landing site. The production classifier is a frozen DINOv3 ViT-L/16 backbone with a trained linear head, about 300M parameters and a 1.2 GB checkpoint, which takes roughly one second per image on a CPU.*

*Operators at a landing site usually have no GPU, so the size of the backbone is a product question as much as an accuracy question.*

## Aim

*Train a linear head on each of six pretrained backbones with one identical recipe, then export every model in the production format so the Fishora evaluation suite can compare them on the same robustness slices and on CPU latency.*

*This notebook only trains and exports. The comparison on hard slices (background removed, background swapped, corruptions, field photos, out-of-distribution images) runs locally through `evaluation/cv/cv_suite.py`, because those slices depend on locally generated fish masks.*

## Metric

***Macro F1-score** averages the per-class F1 over the eleven classes with equal weight. It is appropriate here because the classes are imbalanced (gembolo has 148 test images, senangin has 36), so plain accuracy would let the large classes hide the small ones.*

$$\text{Macro-F1} = \frac{1}{K} \sum_{k=1}^{K} \frac{2 \cdot P_k \cdot R_k}{P_k + R_k}$$

***Expected Calibration Error (ECE)** is reported alongside, because the production pipeline uses the softmax confidence to decide when a human must verify a prediction.*

$$\text{ECE} = \sum_{b=1}^{B} \frac{|S_b|}{N} \left| \text{acc}(S_b) - \text{conf}(S_b) \right|$$

## Dataset

*Data sourced from Kaggle, specifically the Fishora dataset at: https://www.kaggle.com/datasets/athillazaidan/fishora*

*The dataset contains 5,628 photographs of eleven Indonesian fish species from two source families (Roboflow and Fish-gres), split by specimen group into 3,932 train, 847 validation, and 849 test images so that no specimen appears in more than one split.*

*The dataset is used to train and select each linear head (train and validation) and to report in-distribution test metrics.*

```
fishora/
├── cleaned/
|    ├── bandeng/
|    ├── ...
|    └── tuna/
└── splits/
     ├── train.csv
     ├── val.csv
     └── test.csv
```

## Approach: Same-Recipe Linear Probe

*A linear probe freezes a pretrained backbone and trains only its final linear layer. Because every backbone gets the same head, data, augmentation, optimiser, and schedule, a difference in results measures the quality of the backbone features, not the training recipe.*

*The recipe is copied from the notebook that produced the production model (AdamW, learning rate 5e-4, weight decay 0.05, label smoothing 0.1, square-root inverse class weights, cosine schedule over 10 epochs, early stopping on validation macro-F1, then temperature scaling on validation). ViT-L is retrained with it too, so the production model can be separated from the effect of retraining.*

```
Image 256x256 (resize + pad)
    |
    +-- Frozen backbone (eval mode)
         |
    Pre-logit features [384 .. 1280]
         |
    Trainable linear head -> 11 logits -> softmax(logits / T)
```

*One deliberate change from the original recipe: the backbone stays in eval mode during training. For the ViT backbones this is identical to the original, since they have no batch normalisation and no active drop path. For the CNN backbones it keeps BatchNorm running statistics frozen, which is what a linear probe requires.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Enable a GPU (T4 or P100) and internet access in the Kaggle session settings before running. Each finished backbone is written to disk immediately, so after a disconnect a re-run of all cells skips the backbones that are already exported. Expected runtime on a single T4 is about 45 minutes, most of it spent on ViT-L.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q -U "timm>=1.0.20"

## Import Libraries

In [ ]:
import gc
import json
import os
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

print('timm', timm.__version__, '| torch', torch.__version__)

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

## Settings

All paths, backbones, and hyperparameters are centralised here. The recipe values match the production training notebook exactly. `BACKBONES` maps a short name to the timm model name that is also written into each export.

In [ ]:
class Settings:
    SEED       = 42
    _ON_KAGGLE = Path('/kaggle/input').exists()

    DATA_ROOT  = None
    OUTPUT_DIR = Path('/kaggle/working/fishora_linear_probe') if _ON_KAGGLE else Path('../kaggle_outputs/linear_probe/fishora_linear_probe')

    BACKBONES = {
        'vit_l_dinov3':     'vit_large_patch16_dinov3.lvd1689m',
        'vit_b_dinov3':     'vit_base_patch16_dinov3.lvd1689m',
        'vit_s_dinov3':     'vit_small_patch16_dinov3.lvd1689m',
        'convnext_t_dinov3': 'convnext_tiny.dinov3_lvd1689m',
        'efficientnet_b0':  'efficientnet_b0.ra_in1k',
        'mobilenetv3_l':    'mobilenetv3_large_100.ra_in1k',
    }

    IMG_SIZE          = 256
    BATCH_SIZE        = 16
    NUM_WORKERS       = 4
    EPOCHS            = 10
    EARLY_STOP        = 3
    HEAD_LR           = 5e-4
    WEIGHT_DECAY      = 0.05
    LABEL_SMOOTHING   = 0.10
    GRAD_CLIP         = 1.0
    AMP               = True
    ABSTAIN_THRESHOLD = 0.0
    ECE_BINS          = 15

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = CFG.AMP and DEVICE.type == 'cuda'
print('device:', DEVICE, '| output:', CFG.OUTPUT_DIR)

## Load Dataset

The dataset root is the first directory under `/kaggle/input` that contains both `cleaned/` and `splits/`. Paths in the split files are relative to that root.

In [ ]:
def discover_root():
    if CFG.DATA_ROOT:
        return Path(CFG.DATA_ROOT)
    search = Path('/kaggle/input') if CFG._ON_KAGGLE else Path('../data/dataset').resolve().parent
    candidates = [p for p in search.rglob('*') if p.is_dir() and (p / 'cleaned').is_dir() and (p / 'splits').is_dir()]
    if not candidates:
        raise FileNotFoundError('No directory with cleaned/ and splits/ found. Set Settings.DATA_ROOT.')
    return sorted(candidates, key=lambda p: len(str(p)))[0]

ROOT = discover_root()

def load_split(name):
    df = pd.read_csv(ROOT / 'splits' / f'{name}.csv')
    df['resolved_path'] = [str(ROOT / p) for p in df['clean_path']]
    missing = [p for p in df['resolved_path'] if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(f'{len(missing)} {name} images missing, first: {missing[0]}')
    return df

train_df, val_df, test_df = (load_split(s) for s in ('train', 'val', 'test'))
CLASSES = sorted(train_df['normalized_label'].unique())
C2I = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print('root:', ROOT)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}  Classes: {NUM_CLASSES}')

---

# Data Pipeline <a name="3"></a>

---

Preprocessing follows the production wrapper: the image is scaled to fit 256x256 and padded with the dataset mean colour, so the fish is never cropped. Training adds the same light augmentation as the production notebook. All six backbones use ImageNet mean and standard deviation, which was verified from their timm pretrained configs.

## Transforms

`ResizePad` is copied from the production `inference.py`. The fill colour is the ImageNet mean in 0-255 units, matching `fill_rgb` in the production export.

In [ ]:
class ResizePad:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert('RGB')
        w, h = image.size
        scale = min(self.size / w, self.size / h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        image = image.resize((nw, nh), Image.Resampling.BICUBIC)
        canvas = Image.new('RGB', (self.size, self.size), self.fill)
        canvas.paste(image, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)
FILL_RGB = tuple(int(round(x * 255)) for x in MEAN)

train_tf = transforms.Compose([
    ResizePad(CFG.IMG_SIZE, FILL_RGB),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.94, 1.04), fill=FILL_RGB),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    ResizePad(CFG.IMG_SIZE, FILL_RGB),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
print('fill_rgb:', FILL_RGB)

## Datasets and Class Weights

Class weights are the square root of inverse class frequency, normalised to mean one, as in the production recipe. They soften the imbalance without letting the smallest classes dominate the loss.

In [ ]:
class FishDataset(Dataset):
    def __init__(self, df, tf):
        self.paths = df['resolved_path'].tolist()
        self.labels = [C2I[c] for c in df['normalized_label']]
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as img:
            x = self.tf(img.convert('RGB'))
        return x, self.labels[i]

def make_loader(df, tf, shuffle):
    return DataLoader(
        FishDataset(df, tf), batch_size=CFG.BATCH_SIZE, shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
        generator=torch.Generator().manual_seed(CFG.SEED),
    )

train_loader = make_loader(train_df, train_tf, shuffle=True)
val_loader   = make_loader(val_df, eval_tf, shuffle=False)
test_loader  = make_loader(test_df, eval_tf, shuffle=False)

counts = train_df['normalized_label'].value_counts().reindex(CLASSES).to_numpy(dtype=np.float64)
weights = 1.0 / np.sqrt(counts)
weights = weights / weights.mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)
pd.DataFrame({'train_count': counts.astype(int), 'weight': weights.round(3)}, index=CLASSES)

---

# Training Recipe <a name="4"></a>

---

The functions in this section are shared by every backbone, which is what makes the comparison a same-recipe comparison. Only the backbone name changes between runs.

## Model Construction

`build_model` mirrors the production wrapper line for line: create the timm model with an 11-class head, then request CLS-token pooling. The CNN backbones reject `global_pool='token'` and keep average pooling, exactly as they would inside `inference.py`. Every parameter except the final linear layer is frozen.

In [ ]:
def build_model(model_name, pretrained):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=NUM_CLASSES)
    try:
        model.reset_classifier(num_classes=NUM_CLASSES, global_pool='token')
    except Exception:
        pass
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True
    return model

def head_logits(model, x):
    with torch.no_grad():
        feats = model.forward_head(model.forward_features(x), pre_logits=True)
    return model.get_classifier()(feats.float())

## Train and Evaluate

The backbone forward pass runs under `no_grad` and in eval mode, and only the head receives gradients. Evaluation returns raw logits so that temperature scaling and calibration can be computed afterwards without another pass over the images.

In [ ]:
def train_one_epoch(model, optimizer, scaler, criterion):
    model.eval()
    head = model.get_classifier()
    head.train()
    total, n = 0.0, 0
    for x, y in tqdm(train_loader, leave=False, desc='train'):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
            logits = head_logits(model, x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(head.parameters(), CFG.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(y)
        n += len(y)
    return total / n

@torch.no_grad()
def collect_logits(model, loader):
    model.eval()
    logits, labels = [], []
    for x, y in tqdm(loader, leave=False, desc='eval'):
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
            out = model(x.to(DEVICE, non_blocking=True))
        logits.append(out.float().cpu())
        labels.append(y)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()

## Metrics and Temperature Scaling

Temperature is fitted on validation logits by minimising negative log-likelihood, as in the production notebook. ECE uses 15 equal-width confidence bins.

In [ ]:
def softmax_t(logits, temperature):
    z = torch.tensor(logits, dtype=torch.float32) / temperature
    return torch.softmax(z, dim=1).numpy()

def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float32)
    y = torch.tensor(labels, dtype=torch.long)
    log_t = nn.Parameter(torch.zeros(1))
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=100, line_search_fn='strong_wolfe')

    def closure():
        opt.zero_grad()
        loss = nn.functional.cross_entropy(z / log_t.exp().clamp(0.05, 20.0), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.detach().exp().clamp(0.05, 20.0))

def ece_score(probs, labels, bins=CFG.ECE_BINS):
    conf = probs.max(axis=1)
    correct = probs.argmax(axis=1) == labels
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def summarize(probs, labels):
    pred = probs.argmax(axis=1)
    nll = -np.log(np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)).mean()
    return {
        'top1': float(accuracy_score(labels, pred)),
        'macro_f1': float(f1_score(labels, pred, average='macro')),
        'nll': float(nll),
        'ece': ece_score(probs, labels),
        'mean_conf': float(probs.max(axis=1).mean()),
    }

## Export in Production Format

Each export holds the full model state dict, an `inference_config.json` with the same keys as the production config, and the production `inference.py` unchanged. A parity check then reloads the export through that wrapper and confirms it reproduces the notebook's test probabilities, so the evaluation suite measures the same model that was trained here.

In [ ]:
INFERENCE_PY = r'''import json
from pathlib import Path
import torch
import timm
from PIL import Image, ImageOps
from torchvision import transforms

class ResizePad:
    def __init__(self, size, fill):
        self.size = int(size)
        self.fill = tuple(fill)

    def __call__(self, image):
        image = ImageOps.exif_transpose(image).convert("RGB")
        w, h = image.size
        scale = min(self.size / w, self.size / h)
        nw = max(1, int(round(w * scale)))
        nh = max(1, int(round(h * scale)))
        image = image.resize((nw, nh), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(image, ((self.size - nw) // 2, (self.size - nh) // 2))
        return canvas

class FishoraClassifier:
    def __init__(self, export_dir, device=None):
        export_dir = Path(export_dir)
        with open(export_dir / "inference_config.json", "r", encoding="utf-8") as f:
            self.cfg = json.load(f)

        self.device = torch.device(device if device else ("cuda" if torch.cuda.is_available() else "cpu"))

        self.model = timm.create_model(
            self.cfg["model_name"],
            pretrained=False,
            num_classes=self.cfg["num_classes"],
        )
        try:
            self.model.reset_classifier(num_classes=self.cfg["num_classes"], global_pool="token")
        except Exception:
            pass

        state = torch.load(export_dir / "model_state_dict.pt", map_location="cpu", weights_only=True)
        self.model.load_state_dict(state)
        self.model = self.model.to(self.device).eval()

        self.transform = transforms.Compose([
            ResizePad(self.cfg["img_size"], self.cfg["fill_rgb"]),
            transforms.ToTensor(),
            transforms.Normalize(self.cfg["mean"], self.cfg["std"]),
        ])

    @torch.inference_mode()
    def predict(self, image, top_k=3):
        if isinstance(image, (str, Path)):
            image = Image.open(image)

        x = self.transform(image.convert("RGB")).unsqueeze(0).to(self.device)
        logits = self.model(x)
        probs = torch.softmax(logits.float() / float(self.cfg["temperature"]), dim=1)[0]

        k = min(top_k, len(self.cfg["classes"]))
        values, indices = probs.topk(k)
        candidates = [
            {"label": self.cfg["classes"][int(idx)], "confidence": float(value)}
            for value, idx in zip(values.cpu(), indices.cpu())
        ]

        best = candidates[0]
        threshold = float(self.cfg["abstain_threshold"])

        return {
            "status": (
                "confident_prediction"
                if best["confidence"] >= threshold
                else "low_confidence_human_verification_required"
            ),
            "prediction": best,
            "top_candidates": candidates,
            "threshold": threshold,
        }
'''

def write_export(model, model_name, export_dir, temperature, val_metrics, test_metrics):
    export_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), export_dir / 'model_state_dict.pt')
    (export_dir / 'inference.py').write_text(INFERENCE_PY, encoding='utf-8')
    config = {
        'library': 'timm',
        'model_name': model_name,
        'num_classes': NUM_CLASSES,
        'classes': CLASSES,
        'class_to_idx': C2I,
        'img_size': CFG.IMG_SIZE,
        'mean': list(MEAN),
        'std': list(STD),
        'fill_rgb': list(FILL_RGB),
        'temperature': temperature,
        'abstain_threshold': CFG.ABSTAIN_THRESHOLD,
        'training_strategy': 'frozen_backbone_head_only',
        'recipe': 'fishora_linear_probe_v1',
        'validation_metrics': val_metrics,
        'test_metrics': test_metrics,
    }
    (export_dir / 'inference_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')

def parity_check(export_dir, expected_probs, n=8):
    ns = {}
    exec(compile(INFERENCE_PY, str(export_dir / 'inference.py'), 'exec'), ns)
    clf = ns['FishoraClassifier'](export_dir, device=str(DEVICE))
    worst = 0.0
    for i in range(n):
        out = clf.predict(test_df['resolved_path'].iloc[i], top_k=NUM_CLASSES)
        got = np.zeros(NUM_CLASSES)
        for c in out['top_candidates']:
            got[C2I[c['label']]] = c['confidence']
        worst = max(worst, float(np.abs(got - expected_probs[i]).max()))
    del clf
    return worst

---

# Linear Probe Runs <a name="5"></a>

---

Each backbone is trained, calibrated, evaluated, exported, and parity-checked in turn, then released from GPU memory. A backbone whose `result.json` already exists is loaded from disk instead of retrained, which makes the loop safe to resume.

In [ ]:
def run_backbone(short, model_name):
    run_dir = CFG.OUTPUT_DIR / short
    result_path = run_dir / 'result.json'
    if result_path.exists():
        print(f'[{short}] already done, loading result')
        return json.loads(result_path.read_text())

    seed_everything(CFG.SEED)
    run_dir.mkdir(parents=True, exist_ok=True)
    model = build_model(model_name, pretrained=True).to(DEVICE)
    head = model.get_classifier()
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE), label_smoothing=CFG.LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(head.parameters(), lr=CFG.HEAD_LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

    history, best_f1, best_state, stale = [], -1.0, None, 0
    t0 = time.time()
    for epoch in range(1, CFG.EPOCHS + 1):
        train_loss = train_one_epoch(model, optimizer, scaler, criterion)
        val_logits, val_y = collect_logits(model, val_loader)
        val_f1 = f1_score(val_y, val_logits.argmax(axis=1), average='macro')
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_macro_f1': val_f1})
        print(f'[{short}] epoch {epoch:02d} loss={train_loss:.4f} val_f1={val_f1:.4f}')
        if val_f1 > best_f1 + 1e-4:
            best_f1, stale = val_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        else:
            stale += 1
            if stale >= CFG.EARLY_STOP:
                print(f'[{short}] early stop')
                break
    train_minutes = (time.time() - t0) / 60
    head.load_state_dict(best_state)

    val_logits, val_y = collect_logits(model, val_loader)
    test_logits, test_y = collect_logits(model, test_loader)
    temperature = fit_temperature(val_logits, val_y)
    val_metrics = summarize(softmax_t(val_logits, temperature), val_y)
    test_probs = softmax_t(test_logits, temperature)
    test_metrics = summarize(test_probs, test_y)
    per_class_f1 = f1_score(test_y, test_probs.argmax(axis=1), average=None, labels=range(NUM_CLASSES))

    export_dir = run_dir / 'export'
    write_export(model, model_name, export_dir, temperature, val_metrics, test_metrics)
    np.save(run_dir / 'test_logits.npy', test_logits)
    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    parity = parity_check(export_dir, test_probs)

    result = {
        'backbone': short,
        'model_name': model_name,
        'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
        'feature_dim': head.in_features,
        'export_mb': (export_dir / 'model_state_dict.pt').stat().st_size / 2**20,
        'epochs_run': len(history),
        'train_minutes': train_minutes,
        'temperature': temperature,
        'parity_max_abs_diff': parity,
        **{f'val_{k}': v for k, v in val_metrics.items()},
        **{f'test_{k}': v for k, v in test_metrics.items()},
        'test_per_class_f1': dict(zip(CLASSES, map(float, per_class_f1))),
    }
    result_path.write_text(json.dumps(result, indent=2))
    print(f"[{short}] test macro-F1={test_metrics['macro_f1']:.4f} ECE={test_metrics['ece']:.4f} "
          f'T={temperature:.4f} parity={parity:.2e} ({train_minutes:.1f} min)')

    del model, optimizer, scaler, head
    gc.collect()
    torch.cuda.empty_cache()
    return result

The loop below runs every backbone in `Settings.BACKBONES`. Smaller backbones go last in the dictionary only for readability; the order does not affect results because every run reseeds.

In [ ]:
results = [run_backbone(short, name) for short, name in CFG.BACKBONES.items()]
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'test_per_class_f1'} for r in results])
results_df.to_csv(CFG.OUTPUT_DIR / 'results.csv', index=False)
results_df

---

# Results <a name="6"></a>

---

These are in-distribution numbers on the held-out test split. They are expected to be close to saturated for every backbone, because the test split shares photographing conditions with training. The decisive comparison is on the hard slices in the local evaluation suite, so this section is a sanity check, not a ranking.

## Summary Table

Parity below `1e-2` confirms that each export, loaded through the production wrapper in fp32, reproduces the notebook's fp16 probabilities. The tolerance is loose because a temperature below one amplifies small fp16 logit differences.

In [ ]:
cols = ['backbone', 'params_m', 'export_mb', 'feature_dim', 'epochs_run', 'train_minutes',
        'temperature', 'test_top1', 'test_macro_f1', 'test_ece', 'test_nll', 'parity_max_abs_diff']
summary = results_df[cols].sort_values('params_m', ascending=False).reset_index(drop=True)
summary.style.format({c: '{:.4f}' for c in cols[6:]} | {'params_m': '{:.1f}', 'export_mb': '{:.0f}', 'train_minutes': '{:.1f}'})

## Per-Class F1

Per-class F1 shows whether any backbone already struggles on a specific species pair in distribution, for example the two tilapia (mujair and nila) or the two croakers (gulamah and gelama bunga).

In [ ]:
per_class = pd.DataFrame({r['backbone']: r['test_per_class_f1'] for r in results}).T
fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(per_class.to_numpy(), vmin=per_class.to_numpy().min(), vmax=1.0, cmap='viridis')
ax.set_xticks(range(NUM_CLASSES), per_class.columns, rotation=45, ha='right')
ax.set_yticks(range(len(per_class)), per_class.index)
for i in range(per_class.shape[0]):
    for j in range(per_class.shape[1]):
        ax.text(j, i, f'{per_class.iat[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white')
fig.colorbar(im, ax=ax, label='F1')
ax.set_title('Test per-class F1 by backbone (in-distribution)')
fig.tight_layout()
fig.savefig(CFG.OUTPUT_DIR / 'per_class_f1.png', dpi=150)
plt.show()

#### Insights

> Fill in after the run: which backbones saturate the in-distribution test split, and whether any class pair already separates the backbones before the hard slices are applied.

> A near-identical table across backbones is the expected outcome and is itself evidence that the in-distribution test split cannot rank them, which motivates the slice-based comparison.

---

# Export <a name="7"></a>

---

Each backbone is zipped separately so that a single model can be downloaded without the 1.2 GB ViT-L archive. Unzip an archive into `data/linear_probe/` in the repository and point the evaluation suite at its `export/` directory.

In [ ]:
archives = []
for short in CFG.BACKBONES:
    base = CFG.OUTPUT_DIR.parent / f'fishora_lp_{short}'
    archives.append(shutil.make_archive(str(base), 'zip', root_dir=CFG.OUTPUT_DIR, base_dir=short))
shutil.copy(CFG.OUTPUT_DIR / 'results.csv', CFG.OUTPUT_DIR.parent / 'fishora_lp_results.csv')
for a in archives:
    print(f'{Path(a).name:40s} {Path(a).stat().st_size / 2**20:8.1f} MB')